In [ ]:
import logging
import random

import numpy as np
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

from vasae import VASAE
from vasae.data.dataset import GPT2LayerActivations

logging.basicConfig(
    format="[%(levelname)s] %(asctime)s %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
)

/home/zkr/miniconda3/envs/internlm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zkr/miniconda3/envs/internlm/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/zkr/miniconda3/envs/internlm/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [2]:
class CFG:
    seed = 42
    k = 20  # topk sparsity for SAE

    meta_path = "/mnt/data/gpt2_activations/meta.json"
    layer_name = "transformer.h.5.mlp.c_proj"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "gpt2"
    save_dir = "out"
    save_filename = "loss_data.pkl"
    num_epochs = 20


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [3]:
args = CFG()
set_seed(args.seed)
dataset = GPT2LayerActivations(args.meta_path, args.layer_name)

tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
gpt2 = GPT2LMHeadModel.from_pretrained('gpt2')
lm_head = gpt2.lm_head

/home/zkr/miniconda3/envs/internlm/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
h_l = dataset[80:81]
logits = lm_head(h_l)
res = tokenizer.decode(torch.argmax(logits[0,-1,:]))
print(res)

,


/home/zkr/SAE/dataset.py:54: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  return torch.from_numpy(arr)


In [5]:
sae = VASAE(
    topk=args.k,
    emb_weight=gpt2.transformer.wte.weight,
).to(args.device)
sae.load_state_dict(torch.load("out/sae.pth"))

<All keys matched successfully>

In [6]:
h_l = h_l.to(args.device)
with torch.no_grad():
    sae_activations = sae.encoder(h_l)
    topk, indices = torch.topk(sae_activations[0,-1,:], args.k)
for token_id, w in zip(indices, topk):
    print(tokenizer.decode(token_id), '\t', w.item())

 more 	 0.5859609246253967
 The 	 0.5403321981430054
 now 	 0.5152741074562073
. 	 0.49310386180877686
 has 	 0.43995869159698486
 be 	 0.4047691226005554
 new 	 0.37210848927497864
 B 	 0.2919258773326874
 w 	 0.263248085975647
 honor 	 0.25685036182403564
 gel 	 0.1908201277256012
C 	 0.14711353182792664
 and 	 0.12515372037887573
 ACT 	 0.00932714156806469
 to 	 -0.01847800426185131
uphem 	 -0.22261926531791687
fleet 	 -0.22645920515060425
hai 	 -0.23854410648345947
cheon 	 -0.24718263745307922
 quadru 	 -0.2621645927429199


In [7]:
with torch.no_grad():
    decoded, _ = sae(h_l.to(args.device))
nn.MSELoss()(h_l.to(args.device), decoded) # loss

tensor(0.4130, device='cuda:0')